# PII Detection - Dev Log

## Objetivo e papel no pipeline

Este modulo (`core/pii_detection/`) e o **primeiro estagio de analise** do
AthenaGov AI: recebe texto livre (ex. prompt de usuario, documento, log de
conversa com um LLM) e identifica dados pessoais e sensiveis conforme o
Art. 5 da LGPD, antes que esse texto siga para o Policy Engine, o Trust
Score ou vire evidencia num RIPD.

A saida (`PIIDetectionResult`, contrato definido em `shared/schemas.py`) e
consumida por outros modulos do pipeline (Policy Engine decide `ALLOW` /
`DENY` / `REQUIRES_HUMAN_REVIEW` com base nos achados; Audit Logs registra
o evento `PII_SCAN`; RIPD Engine usa os achados para preencher
`data_categories` do relatorio).

**Assinatura publica:**

```python
def detect(text: str) -> PIIDetectionResult: ...
```


## Decisoes de design

**Por que regex-first, e nao NER-first?**

- **Determinismo e custo zero**: CPF, CNPJ, e-mail, telefone e CEP tem
  formato sintatico bem definido. Regex com validacao de digito
  verificador (CPF/CNPJ) e mais confiavel e mais rapida que um modelo de
  NLP para esses casos, e nao depende de download de modelo, GPU, ou
  disponibilidade de rede.
- **Auditabilidade**: cada achado do caminho regex e rastreavel a um
  padrao explicito no codigo - importante num produto de compliance, onde
  "por que isso foi marcado como PII" precisa ter resposta deterministica.
- **NER como enriquecimento, nunca dependencia**: nomes proprios em texto
  livre nao tem um padrao sintatico confiavel (diferente de CPF/e-mail), e
  e ai que um NER real (spaCy `pt_core_news_sm`) ajudaria mais. Mas exigir
  spaCy tornaria o caminho critico fragil (download de modelo, tempo de
  carregamento). Por isso o spaCy e chamado em `try/except` dentro de
  `_match_names_spacy`: se falhar ou nao estiver instalado, o resultado
  cai de volta para a heuristica regex sozinha - **nenhum teste depende
  do spaCy estar disponivel**.

**Limitacoes conhecidas da heuristica de nome proprio (regex, sem NER):**

- Falso positivo: qualquer sequencia de 2+ palavras capitalizadas fora de
  inicio de frase pode ser confundida com nome (ex. titulos de secao,
  siglas de orgaos, nomes de produtos). A lista de stopwords em portugues
  reduz mas nao elimina isso.
- Falso negativo: nomes de uma palavra so (raro, mas existe), nomes que
  aparecem logo no inicio da frase (deliberadamente ignorados para
  reduzir falso positivo), e apelidos/nomes sem capitalizacao correta.
- A heuristica **nao valida** que a sequencia capturada e de fato um nome
  humano - e puramente sintatica.

**Limitacoes da deteccao de dado sensivel:**

- E por **palavra-chave/frase**, nao por NLP real de intencao ou
  contexto. "Ele NAO e evangelico" ainda dispara `SENSITIVE_RELIGION`,
  porque o motor nao faz analise de negacao. Isso e uma limitacao
  deliberada do V1 (documentada, nao escondida) - reduzir falso negativo
  (nao deixar passar mencao sensivel) tem prioridade sobre reduzir falso
  positivo aqui, dado o contexto de compliance.

**RG sem digito verificador:** diferente de CPF/CNPJ, o formato de RG
varia por estado emissor e nao tem um algoritmo de validacao universal.
Por isso a deteccao de RG exige a palavra-chave "RG" proxima ao numero
(janela de contexto), em vez de confiar so no formato dos digitos - que e
identico ao formato de CPF em varios estados.


In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

from core.pii_detection.detector import detect

print("Modulo carregado:", detect.__module__)
print("Assinatura:", detect.__doc__.splitlines()[0])


Modulo carregado: core.pii_detection.detector
Assinatura: Detecta dados pessoais e sensíveis em texto livre em português.


In [1]:
texto_1 = (
    "Cliente Joao da Silva, CPF 111.444.777-35, e-mail joao.silva@empresa.com.br, "
    "telefone (11) 91234-5678, mora no CEP 01310-100."
)
resultado_1 = detect(texto_1)
print("Texto:", texto_1)
print()
print(resultado_1.model_dump_json(indent=2))


Texto: Cliente Joao da Silva, CPF 111.444.777-35, e-mail joao.silva@empresa.com.br, telefone (11) 91234-5678, mora no CEP 01310-100.

{
  "findings": [
    {
      "entity_type": "CPF",
      "text_span": "111.444.777-35",
      "start": 27,
      "end": 41,
      "category": "personal",
      "confidence": 0.98
    },
    {
      "entity_type": "EMAIL",
      "text_span": "joao.silva@empresa.com.br",
      "start": 50,
      "end": 75,
      "category": "personal",
      "confidence": 0.97
    },
    {
      "entity_type": "TELEFONE",
      "text_span": "(11) 91234-5678",
      "start": 86,
      "end": 101,
      "category": "personal",
      "confidence": 0.85
    },
    {
      "entity_type": "CEP",
      "text_span": "01310-100",
      "start": 115,
      "end": 124,
      "category": "personal",
      "confidence": 0.9
    }
  ],
  "has_sensitive_data": false,
  "summary": "4 achado(s) — CPF=1, EMAIL=1, TELEFONE=1, CEP=1"
}


In [1]:
texto_2 = (
    "Apresentou RG 23.456.789-1 no balcao de atendimento. "
    "Data de nascimento: 02/11/1985, conforme cadastro civil."
)
resultado_2 = detect(texto_2)
print("Texto:", texto_2)
print()
print(resultado_2.model_dump_json(indent=2))


Texto: Apresentou RG 23.456.789-1 no balcao de atendimento. Data de nascimento: 02/11/1985, conforme cadastro civil.

{
  "findings": [
    {
      "entity_type": "RG",
      "text_span": "23.456.789-1",
      "start": 14,
      "end": 26,
      "category": "personal",
      "confidence": 0.85
    },
    {
      "entity_type": "DATA_NASCIMENTO",
      "text_span": "02/11/1985",
      "start": 73,
      "end": 83,
      "category": "personal",
      "confidence": 0.85
    }
  ],
  "has_sensitive_data": false,
  "summary": "2 achado(s) — RG=1, DATA_NASCIMENTO=1"
}


In [1]:
texto_3 = (
    "Durante a consulta, o paciente relatou diagnostico de depressao. "
    "Ele tambem mencionou sua filiacao sindical e disse ser evangelico."
)
resultado_3 = detect(texto_3)
print("Texto:", texto_3)
print()
print(resultado_3.model_dump_json(indent=2))
print()
print("has_sensitive_data:", resultado_3.has_sensitive_data)


Texto: Durante a consulta, o paciente relatou diagnostico de depressao. Ele tambem mencionou sua filiacao sindical e disse ser evangelico.

{
  "findings": [
    {
      "entity_type": "SENSITIVE_HEALTH",
      "text_span": "diagnostico",
      "start": 39,
      "end": 50,
      "category": "sensitive",
      "confidence": 0.65
    },
    {
      "entity_type": "SENSITIVE_HEALTH",
      "text_span": "depressao",
      "start": 54,
      "end": 63,
      "category": "sensitive",
      "confidence": 0.65
    },
    {
      "entity_type": "SENSITIVE_POLITICAL_UNION",
      "text_span": "filiacao sindical",
      "start": 90,
      "end": 107,
      "category": "sensitive",
      "confidence": 0.65
    },
    {
      "entity_type": "SENSITIVE_RELIGION",
      "text_span": "evangelico",
      "start": 120,
      "end": 130,
      "category": "sensitive",
      "confidence": 0.65
    }
  ],
  "has_sensitive_data": true,
  "summary": "4 achado(s) — SENSITIVE_HEALTH=2, SENSITIVE_POLITICAL_UNI

In [1]:
# Mini avaliacao ilustrativa (NAO e um benchmark formal) - 11 casos
# manualmente rotulados, so para dar um numero honesto de FP/FN observado
# nos proprios exemplos deste dev-log. Nao generaliza para producao.

casos = [
    ("O CPF do cliente e 111.444.777-35.", ["CPF"], []),
    ("Email para contato: ana.souza@dominio.com.br", ["EMAIL"], []),
    ("Ligue (21) 98765-4321 para confirmar.", ["TELEFONE"], []),
    ("CEP 04567-000 fica na zona sul.", ["CEP"], []),
    ("Apresentou RG 23.456.789-1 na portaria.", ["RG"], []),
    ("Data de nascimento: 02/11/1985.", ["DATA_NASCIMENTO"], []),
    ("O relatorio foi assinado por Mariana Costa Lima na sexta-feira.", ["NOME"], []),
    ("Ela relatou possuir diagnostico de diabetes.", ["SENSITIVE_HEALTH"], []),
    ("O codigo do pedido e 04567000 sem nenhum outro dado.", [], ["CEP", "CPF"]),
    ("A reuniao foi marcada para 02/11/1985 na sede da empresa.", [], ["DATA_NASCIMENTO"]),
    ("Segundo Relatorio Anual, as vendas cresceram.", [], ["NOME"]),
]

total_checks = 0
acertos = 0
falhas = []

for texto, esperados, proibidos in casos:
    r = detect(texto)
    tipos_encontrados = {f.entity_type for f in r.findings}
    for tipo in esperados:
        total_checks += 1
        ok = tipo in tipos_encontrados
        acertos += int(ok)
        if not ok:
            falhas.append(("FALSO NEGATIVO", texto, tipo))
    for tipo in proibidos:
        total_checks += 1
        ok = tipo not in tipos_encontrados
        acertos += int(ok)
        if not ok:
            falhas.append(("FALSO POSITIVO", texto, tipo))

print(f"Checagens: {total_checks} | Corretas: {acertos} | Taxa de acerto: {acertos/total_checks:.1%}")
print()
if falhas:
    print("Falhas observadas:")
    for tipo_falha, texto, entidade in falhas:
        print(f"  - {tipo_falha}: entidade={entidade!r} texto={texto!r}")
else:
    print("Nenhuma falha nos 11 casos rotulados manualmente neste mini-conjunto.")


Checagens: 12 | Corretas: 12 | Taxa de acerto: 100.0%

Nenhuma falha nos 11 casos rotulados manualmente neste mini-conjunto.


## Suite de testes

Executando a suite pytest real via `subprocess`, a partir da raiz do repo,
com o Python do venv do projeto
(`C:/Users/Yuri_/.venvs/athenagov-ai/Scripts/python.exe`).


In [1]:
import subprocess

python_exe = r"C:/Users/Yuri_/.venvs/athenagov-ai/Scripts/python.exe"
proc = subprocess.run(
    [python_exe, "-m", "pytest", "core/pii_detection/tests", "-v"],
    cwd=str(REPO_ROOT),
    capture_output=True,
    text=True,
)
print(proc.stdout[-3000:])
if proc.returncode != 0:
    print("STDERR:", proc.stderr[-2000:])
print("Return code:", proc.returncode)


===
platform win32 -- Python 3.10.8, pytest-9.1.1, pluggy-1.6.0 -- C:\Users\Yuri_\.venvs\athenagov-ai\Scripts\python.exe
cachedir: .pytest_cache
rootdir: G:\Outros computadores\Meu computador\Controle Base\Projetos, Robos e Automação\Projetos Git\Projetos Extras (Portfolio)\LGPD e IA (Terminar)
plugins: anyio-4.14.2, cov-7.1.0
collecting ... collected 25 items

core/pii_detection/tests/test_detector.py::test_detect_returns_pii_detection_result_instance PASSED [  4%]
core/pii_detection/tests/test_detector.py::test_confidence_scores_within_bounds PASSED [  8%]
core/pii_detection/tests/test_detector.py::test_cpf_formatted_and_valid_detected PASSED [ 12%]
core/pii_detection/tests/test_detector.py::test_cpf_plain_digits_valid_checksum_detected_as_cpf_not_phone PASSED [ 16%]
core/pii_detection/tests/test_detector.py::test_cpf_formatted_invalid_checksum_still_flagged_lower_confidence PASSED [ 20%]
core/pii_detection/tests/test_detector.py::test_cnpj_formatted_and_valid_detected PASSED [ 24%]


## Handoff Summary

**Capacidades entregues (V1):**

- Deteccao regex deterministica e 100% offline de: CPF (com validacao de
  digito verificador), CNPJ (idem), RG (via janela de contexto com a
  palavra "RG"), e-mail, telefone BR (com/sem DDD, com/sem `+55`), CEP e
  data de nascimento (via janela de contexto com palavras-chave como
  "data de nascimento", "nasc.", "DN").
- Heuristica leve de nome proprio (sequencia de 2+ palavras capitalizadas
  fora de inicio de frase, filtrada por lista de stopwords em portugues).
- Classificacao de cada achado em `DataCategory.PERSONAL` ou
  `DataCategory.SENSITIVE` conforme LGPD Art. 5, incluindo deteccao por
  palavra-chave de mencoes a saude, biometria, orientacao sexual,
  religiao, etnia/raca e opiniao politica/filiacao sindical.
- Enriquecimento opcional via spaCy (`pt_core_news_sm`) para NER de
  pessoas, chamado em `try/except` - nunca obrigatorio.
- Resolucao de sobreposicao de spans (ex. um numero de 11 digitos com
  digito verificador de CPF valido e classificado como `CPF` e nao como
  `TELEFONE`, mesmo tendo formato ambiguo).

**Assinatura publica exata:**

```python
def detect(text: str) -> PIIDetectionResult:
```

(de `core/pii_detection/detector.py`, reexportada em
`core/pii_detection/__init__.py`; usa `PIIFinding`, `PIIDetectionResult`,
`DataCategory` de `shared/schemas.py` - nenhum tipo redefinido localmente.)

**Taxa de falso positivo/negativo observada:** no mini-conjunto ilustrativo
de 11 casos rotulados manualmente (celula acima, executada de verdade
neste notebook), a taxa de acerto observada e reportada na saida da
celula de avaliacao - **nao e um benchmark formal** nem generaliza para
producao; serve apenas como evidencia honesta e reproduzivel do
comportamento atual do motor sobre um conjunto pequeno e conhecido de
frases.

**Limitacoes (V1):**

- Heuristica de nome nao e NER real - falso positivo com sequencias
  capitalizadas nao-nome (titulos, siglas), falso negativo com nomes de
  uma palavra ou em inicio de frase.
- Classificacao de dado sensivel e por palavra-chave - nao entende
  negacao ("NAO e evangelico" ainda marca `SENSITIVE_RELIGION`) nem
  contexto semantico mais amplo.
- RG so e detectado com a palavra-chave "RG" nas proximidades - RG
  citado sem esse contexto textual nao e capturado (trade-off deliberado
  para nao confundir com CPF).
- Telefone sem DDD e sem hifen explicito nao e detectado (regra
  deliberadamente conservadora para nao gerar excesso de falso positivo
  com qualquer sequencia de 8 digitos).
- Sem suporte a documentos/imagens (OCR) - o escopo V1 e texto livre.

**O que fica para V2 (ver ROADMAP.md):**

- **Sensitive Data Scanner**: evolucao do PII Detection para documentos e
  imagens via OCR (mencionado explicitamente no ROADMAP como item V2).
  TODO explicito - nao implementado neste ciclo.
- NER real (modelo treinado/fine-tuned para nomes em portugues no
  dominio de compliance) para substituir/reforcar a heuristica regex de
  nome.
- Deteccao de negacao/contexto para reduzir falso positivo de dado
  sensivel.
- Validacao de RG por estado (cada UF tem regras proprias de formato e
  digito verificador).
